# Content-Based Filtering: A Comprehensive Guide

---

## Table of Contents

1. **Introduction to Content-Based Filtering**
2. **Core Concepts and Feature Representation**
3. **Types of Content-Based Filtering**
   - Keyword/Boolean-Based Filtering
   - TF-IDF Based Filtering
   - Embedding-Based Filtering
   - Knowledge-Based Filtering
4. **Mathematical Foundations**
   - Term Frequency-Inverse Document Frequency (TF-IDF)
   - Cosine Similarity
   - Euclidean Distance & Pearson Correlation
5. **Algorithms in Detail**
   - TF-IDF + Cosine Similarity Recommender
   - Naive Bayes Classifier
   - k-Nearest Neighbors (k-NN)
   - Decision Tree-Based Filtering
   - Neural Content-Based Filtering
6. **User and Item Profile Construction**
7. **Industrial Examples**
   - Netflix (Movie Recommendations)
   - Spotify (Music Recommendations)
   - Google News (Article Recommendations)
   - Amazon (Product Recommendations)
8. **Evaluation Metrics**
9. **Advantages, Limitations, and Hybrid Approaches**
10. **Summary and Future Directions**

---

## 1. Introduction to Content-Based Filtering

Content-based filtering is one of the foundational approaches in **recommender systems**. Unlike collaborative filtering — which relies on the collective behavior of many users — content-based filtering recommends items by analyzing the **intrinsic attributes** (content features) of items a user has interacted with in the past.

### The Core Idea

The fundamental principle is straightforward:

> *If a user liked Item A, and Item B shares similar attributes with Item A, then the user will likely enjoy Item B.*

### Formal Definition

Given:
- A user $$u$$ with a history of rated/consumed items $$I_u = \{i_1, i_2, \ldots, i_n\}$$
- A set of candidate items $$C = \{c_1, c_2, \ldots, c_m\}$$
- A feature extraction function $$\phi: I \rightarrow \mathbb{R}^d$$ that maps items to a $$d$$-dimensional feature space

The goal is to learn a **user profile** $$\mathbf{p}_u \in \mathbb{R}^d$$ and compute a **utility score**:

$$\text{score}(u, c) = f(\mathbf{p}_u, \phi(c))$$

where $$f$$ is a scoring function (e.g., cosine similarity, dot product, or a learned function).

### Content-Based vs. Collaborative Filtering

| Aspect | Content-Based | Collaborative |
| --- | --- | --- |
| Data required | Item features | User-item interactions |
| Cold-start (new items) | Handles well | Struggles |
| Cold-start (new users) | Needs some history | Struggles |
| Serendipity | Low (filter bubble) | Higher |
| Scalability | Depends on features | Depends on user base |
| Explainability | High | Low |

## 2. Core Concepts and Feature Representation

### 2.1 Item Representation

Every item in a content-based system must be described by a set of features. The choice of features depends on the domain:

| Domain | Item | Features |
| --- | --- | --- |
| Movies | Film | Genre, director, actors, keywords, plot summary |
| Music | Song | Tempo, key, loudness, energy, artist, genre |
| News | Article | Words, topics, entities, source, category |
| E-commerce | Product | Category, brand, price, description, specs |

### 2.2 Feature Extraction Methods

Feature extraction transforms raw item data into numerical vectors:

**Structured Features** (already numerical or categorical):
- Price, rating, duration → direct numerical encoding
- Genre, category → one-hot or multi-hot encoding

**Unstructured Features** (text, images, audio):
- Text → TF-IDF, Word2Vec, BERT embeddings
- Images → CNN features (ResNet, VGG)
- Audio → MFCCs, spectrograms

### 2.3 The Item Profile Vector

An item $$i$$ is represented as a feature vector:

$$\phi(i) = [w_1, w_2, \ldots, w_d]$$

where each $$w_k$$ represents the importance/weight of feature $$k$$ for item $$i$$.

### 2.4 The User Profile Vector

A user profile is typically constructed by aggregating the feature vectors of items the user has liked:

$$\mathbf{p}_u = \frac{1}{|I_u^+|} \sum_{i \in I_u^+} \phi(i)$$

where $$I_u^+$$ is the set of items positively rated by user $$u$$.

More sophisticated approaches use **weighted averages** based on ratings:

$$\mathbf{p}_u = \frac{\sum_{i \in I_u} r_{u,i} \cdot \phi(i)}{\sum_{i \in I_u} r_{u,i}}$$

where $$r_{u,i}$$ is the rating user $$u$$ gave to item $$i$$.

In [0]:
# Setup: Install and import required libraries
%pip install scikit-learn numpy pandas matplotlib seaborn wordcloud -q

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

## 3. Types of Content-Based Filtering

### 3.1 Keyword/Boolean-Based Filtering

**Concept:** The simplest form of content-based filtering. Items are described by a set of keywords, and user preferences are expressed as boolean queries over these keywords.

**How it works:**
- Items are tagged with keywords (e.g., a movie tagged as "action", "sci-fi", "robots")
- User profile = set of preferred keywords
- Matching = set intersection / overlap

**Scoring:**

$$\text{score}(u, i) = \frac{|K_u \cap K_i|}{|K_u \cup K_i|}$$

This is the **Jaccard Similarity** between user keywords $$K_u$$ and item keywords $$K_i$$.

**Industrial Example — Early Yahoo! Directory:**  
Yahoo!'s original web directory (1994-2014) used keyword matching to recommend web pages. Users specified interests via keywords, and pages matching those keywords were surfaced.

**Limitations:**
- No notion of term importance (all keywords weighted equally)
- Cannot capture semantic similarity
- Binary matching loses nuance

---

### 3.2 TF-IDF Based Filtering

**Concept:** Items are represented as TF-IDF vectors derived from their textual descriptions. This captures both the **frequency** of terms and their **discriminative power** across the corpus.

**Industrial Example — Google News:**  
Google News uses TF-IDF (among other signals) to represent article content and match it against user reading histories to recommend personalized news stories.

---

### 3.3 Embedding-Based Filtering

**Concept:** Items are represented using dense, learned vector embeddings (Word2Vec, Doc2Vec, BERT, or domain-specific embeddings). These capture **semantic meaning** beyond surface-level word matching.

**Key Advantage:** "action movie" and "thriller film" would be close in embedding space even without sharing exact words.

**Industrial Example — Spotify:**  
Spotify's music recommendation uses audio embeddings (convolutional neural networks on spectrograms) combined with natural language embeddings from playlist titles and artist descriptions.

---

### 3.4 Knowledge-Based Filtering

**Concept:** Uses explicit domain knowledge (ontologies, taxonomies, knowledge graphs) to understand relationships between item features.

**How it works:**
- Features are connected in a knowledge graph
- Similarity considers not just exact matches but semantic relationships
- E.g., knowing that "Python" is-a "programming language" is-a "technical skill"

**Industrial Example — LinkedIn:**  
LinkedIn's job recommendations use a knowledge graph of skills, industries, and roles. If a user has "PyTorch" experience, the system knows this relates to "deep learning" → "machine learning" → "data science" and can recommend relevant roles.

## 4. Mathematical Foundations

### 4.1 Term Frequency-Inverse Document Frequency (TF-IDF)

TF-IDF is the backbone of text-based content filtering. It quantifies how important a word is to a document within a corpus.

#### Term Frequency (TF)

Measures how frequently a term appears in a document:

$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$

where $$f_{t,d}$$ is the raw count of term $$t$$ in document $$d$$.

**Variants:**
- **Raw count:** $$\text{TF}(t,d) = f_{t,d}$$
- **Log normalization:** $$\text{TF}(t,d) = 1 + \log(f_{t,d})$$
- **Double normalization:** $$\text{TF}(t,d) = 0.5 + 0.5 \cdot \frac{f_{t,d}}{\max_{t' \in d} f_{t',d}}$$

#### Inverse Document Frequency (IDF)

Measures how rare/discriminative a term is across the corpus:

$$\text{IDF}(t, D) = \log \frac{|D|}{|\{d \in D : t \in d\}|}$$

where $$|D|$$ is the total number of documents and the denominator is the count of documents containing term $$t$$.

**Smoothed variant (prevents division by zero):**

$$\text{IDF}(t, D) = \log \frac{|D| + 1}{|\{d \in D : t \in d\}| + 1} + 1$$

#### TF-IDF Score

The final TF-IDF weight combines both:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

**Interpretation:**
- High TF-IDF → term is frequent in this document but rare overall (highly discriminative)
- Low TF-IDF → term is either rare in the document or common across all documents

---

### 4.2 Cosine Similarity

The most widely used similarity measure in content-based filtering. It measures the **angle** between two vectors, ignoring magnitude:

$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_{i=1}^{d} a_i b_i}{\sqrt{\sum_{i=1}^{d} a_i^2} \cdot \sqrt{\sum_{i=1}^{d} b_i^2}}$$

**Properties:**
- Range: $$[-1, 1]$$ for general vectors, $$[0, 1]$$ for TF-IDF (non-negative)
- $$\text{cos} = 1$$ → identical direction (maximally similar)
- $$\text{cos} = 0$$ → orthogonal (no similarity)
- Magnitude-invariant: a short document and a long document about the same topic will still be similar

---

### 4.3 Other Similarity/Distance Metrics

#### Euclidean Distance

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{d} (a_i - b_i)^2}$$

Converted to similarity: $$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{1}{1 + d(\mathbf{a}, \mathbf{b})}$$

#### Pearson Correlation

$$\rho(\mathbf{a}, \mathbf{b}) = \frac{\sum_{i=1}^{d}(a_i - \bar{a})(b_i - \bar{b})}{\sqrt{\sum_{i=1}^{d}(a_i - \bar{a})^2} \cdot \sqrt{\sum_{i=1}^{d}(b_i - \bar{b})^2}}$$

Useful when features have different scales/baselines.

#### Jaccard Similarity (for binary/set features)

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

In [0]:
# ============================================================
# 4.1 TF-IDF: Implementation from Scratch
# ============================================================
# Let's build TF-IDF from first principles to understand the math

import math
from collections import Counter

def compute_tf(document: str) -> dict:
    """Compute Term Frequency for a document."""
    words = document.lower().split()
    word_count = Counter(words)
    total_words = len(words)
    return {word: count / total_words for word, count in word_count.items()}

def compute_idf(corpus: list) -> dict:
    """Compute Inverse Document Frequency across a corpus."""
    N = len(corpus)
    idf = {}
    # Get all unique words
    all_words = set(word for doc in corpus for word in doc.lower().split())
    
    for word in all_words:
        # Count documents containing this word
        doc_count = sum(1 for doc in corpus if word in doc.lower().split())
        idf[word] = math.log((N + 1) / (doc_count + 1)) + 1  # smoothed IDF
    return idf

def compute_tfidf(corpus: list) -> list:
    """Compute TF-IDF for entire corpus."""
    idf = compute_idf(corpus)
    tfidf_corpus = []
    
    for doc in corpus:
        tf = compute_tf(doc)
        tfidf = {word: tf_val * idf[word] for word, tf_val in tf.items()}
        tfidf_corpus.append(tfidf)
    return tfidf_corpus, idf

# Example: Movie descriptions
movie_corpus = [
    "action thriller with explosive car chases and gun fights",
    "romantic comedy about love and relationships in new york",
    "science fiction space adventure with aliens and robots",
    "action adventure with martial arts and sword fighting",
    "romantic drama about love loss and redemption",
    "science fiction thriller with time travel and paradox"
]

movie_titles = [
    "Fast & Furious", "When Harry Met Sally", "Star Wars",
    "Crouching Tiger", "The Notebook", "Interstellar"
]

# Compute TF-IDF
tfidf_results, idf_scores = compute_tfidf(movie_corpus)

# Display results for first movie
print("=" * 60)
print(f"TF-IDF for '{movie_titles[0]}':")
print(f"Description: '{movie_corpus[0]}'")
print("=" * 60)
print(f"\n{'Term':<15} {'TF-IDF Score':<12}")
print("-" * 30)
for word, score in sorted(tfidf_results[0].items(), key=lambda x: -x[1])[:8]:
    print(f"{word:<15} {score:.4f}")

# Compare with sklearn
print("\n" + "=" * 60)
print("Verification with sklearn TfidfVectorizer:")
print("=" * 60)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(movie_corpus)
feature_names = vectorizer.get_feature_names_out()

# Show top terms for first document
doc_tfidf = tfidf_matrix[0].toarray().flatten()
top_indices = doc_tfidf.argsort()[-8:][::-1]
print(f"\n{'Term':<15} {'TF-IDF Score':<12}")
print("-" * 30)
for idx in top_indices:
    if doc_tfidf[idx] > 0:
        print(f"{feature_names[idx]:<15} {doc_tfidf[idx]:.4f}")

In [0]:
# ============================================================
# 4.2 Cosine Similarity: Visualization and Implementation
# ============================================================

def cosine_sim(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

# Compute pairwise similarities for our movie corpus
similarity_matrix = cosine_similarity(tfidf_matrix)

# Create a DataFrame for better visualization
sim_df = pd.DataFrame(
    similarity_matrix,
    index=movie_titles,
    columns=movie_titles
)

print("Pairwise Cosine Similarity Matrix (Movie Descriptions):")
print("=" * 70)
display(sim_df.round(3))

# Visualize as heatmap
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
sns.heatmap(
    sim_df, annot=True, fmt='.3f', cmap='YlOrRd',
    square=True, linewidths=0.5, ax=ax
)
ax.set_title('Content Similarity Matrix (TF-IDF + Cosine Similarity)', fontsize=14)
plt.tight_layout()
plt.show()

# Interpretation
print("\nKey Observations:")
print("-" * 50)
print("- 'Fast & Furious' is most similar to 'Crouching Tiger' (both action/adventure)")
print("- 'When Harry Met Sally' is most similar to 'The Notebook' (both romantic)")
print("- 'Star Wars' is most similar to 'Interstellar' (both sci-fi)")
print("- Cross-genre pairs have low similarity (e.g., Romance vs Sci-fi)")

## 5. Algorithms in Detail

### 5.1 TF-IDF + Cosine Similarity Recommender

This is the most classical content-based filtering algorithm. It works in three phases:

**Phase 1: Item Profiling**
- Extract textual features from items (descriptions, metadata)
- Compute TF-IDF vectors for all items
- Result: Each item $$i$$ is represented as $$\phi(i) \in \mathbb{R}^{|V|}$$ where $$|V|$$ is vocabulary size

**Phase 2: User Profiling**
- Aggregate TF-IDF vectors of items the user has liked/rated highly
- User profile: $$\mathbf{p}_u = \frac{\sum_{i \in I_u^+} r_{u,i} \cdot \phi(i)}{\sum_{i \in I_u^+} r_{u,i}}$$

**Phase 3: Scoring & Ranking**
- For each candidate item $$c$$, compute: $$\text{score}(u, c) = \cos(\mathbf{p}_u, \phi(c))$$
- Return top-$$k$$ items by score

**Complexity:**
- Time: $$O(|C| \cdot |V|)$$ for scoring all candidates
- Space: $$O(|I| \cdot |V|)$$ for storing all item profiles

**Industrial Example — Netflix (Early System):**  
Netflix's early recommendation engine represented movies using TF-IDF vectors of plot summaries, cast/crew metadata, and user-generated tags. When a user watched and rated several Christopher Nolan films, the system would build a user profile emphasizing terms like "mind-bending", "nonlinear", "thriller" and recommend similar films.

In [0]:
# ============================================================
# 5.1 Complete Content-Based Recommender System
# Industrial Example: Netflix-style Movie Recommendations
# ============================================================

class ContentBasedRecommender:
    """
    A complete content-based filtering system using TF-IDF and cosine similarity.
    Models the approach used by services like Netflix for content matching.
    """
    
    def __init__(self, max_features=5000, ngram_range=(1, 2)):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            stop_words='english'
        )
        self.item_profiles = None
        self.item_names = None
        self.item_metadata = None
    
    def fit(self, items_df: pd.DataFrame, text_column: str, name_column: str):
        """Build item profiles from textual descriptions."""
        self.item_names = items_df[name_column].values
        self.item_metadata = items_df
        # Compute TF-IDF matrix
        self.item_profiles = self.vectorizer.fit_transform(items_df[text_column])
        print(f"Built profiles for {self.item_profiles.shape[0]} items")
        print(f"Vocabulary size: {self.item_profiles.shape[1]} features")
        return self
    
    def build_user_profile(self, liked_indices: list, ratings: list = None):
        """
        Construct user profile by aggregating liked item profiles.
        
        p_u = sum(r_i * phi(i)) / sum(r_i)
        """
        if ratings is None:
            ratings = [1.0] * len(liked_indices)
        
        # Weighted average of item profiles
        weighted_sum = np.zeros(self.item_profiles.shape[1])
        total_weight = 0
        
        for idx, rating in zip(liked_indices, ratings):
            item_vec = self.item_profiles[idx].toarray().flatten()
            weighted_sum += rating * item_vec
            total_weight += rating
        
        user_profile = weighted_sum / total_weight
        return user_profile.reshape(1, -1)
    
    def recommend(self, user_profile, n=5, exclude_indices=None):
        """Generate top-N recommendations for a user."""
        # Compute similarity between user profile and all items
        similarities = cosine_similarity(user_profile, self.item_profiles).flatten()
        
        # Exclude already-seen items
        if exclude_indices:
            for idx in exclude_indices:
                similarities[idx] = -1
        
        # Get top-N indices
        top_indices = similarities.argsort()[-n:][::-1]
        
        recommendations = []
        for idx in top_indices:
            recommendations.append({
                'item': self.item_names[idx],
                'similarity_score': similarities[idx],
                'rank': len(recommendations) + 1
            })
        
        return pd.DataFrame(recommendations)
    
    def find_similar_items(self, item_index: int, n: int = 5):
        """Find items most similar to a given item."""
        item_vec = self.item_profiles[item_index]
        similarities = cosine_similarity(item_vec, self.item_profiles).flatten()
        similarities[item_index] = -1  # Exclude self
        
        top_indices = similarities.argsort()[-n:][::-1]
        results = []
        for idx in top_indices:
            results.append({
                'similar_item': self.item_names[idx],
                'similarity': similarities[idx]
            })
        return pd.DataFrame(results)


# ============================================================
# Create a synthetic movie dataset (Netflix-style)
# ============================================================
movies_data = {
    'title': [
        'The Dark Knight', 'Inception', 'Interstellar', 'The Prestige',
        'Pulp Fiction', 'Kill Bill', 'Django Unchained', 'Inglourious Basterds',
        'The Notebook', 'A Walk to Remember', 'Pride and Prejudice', 'La La Land',
        'The Matrix', 'Blade Runner 2049', 'Ex Machina', 'Her',
        'Toy Story', 'Finding Nemo', 'Inside Out', 'Coco',
        'The Godfather', 'Goodfellas', 'The Departed', 'Casino',
        'Avengers Endgame', 'Spider-Man', 'Black Panther', 'Iron Man',
        'Titanic', 'Forrest Gump', 'The Shawshank Redemption', 'Schindlers List'
    ],
    'description': [
        'dark superhero crime thriller batman joker gotham city justice vigilante action intense psychological',
        'mind-bending science fiction dreams layers reality heist thriller psychological cerebral complex',
        'epic space science fiction love time gravity black hole survival emotional father daughter',
        'mystery thriller magic rivalry obsession dark twist victorian era illusionists secrets',
        'crime nonlinear storytelling dark humor violence gangsters hitmen dialogue iconic',
        'martial arts revenge action sword samurai stylized violence strong female protagonist',
        'western revenge slavery action violence historical drama bounty hunter',
        'world war two revenge nazis dark humor violence alternate history action',
        'romantic love story passion summer letters devotion alzheimers emotional tearjerker',
        'teenage romance love faith illness small town emotional touching sad',
        'period romance love class society manners witty british literature adaptation',
        'musical romance love dreams hollywood jazz ambition bittersweet beautiful',
        'science fiction artificial intelligence simulation reality action philosophical cyberpunk martial arts',
        'science fiction dystopian future replicants artificial intelligence visual stunning atmospheric noir',
        'artificial intelligence consciousness robot thriller philosophical isolation turing test',
        'romance artificial intelligence loneliness connection futuristic operating system emotional',
        'animation adventure friendship toys imagination children fun comedy heartwarming',
        'animation underwater adventure family father son ocean fish friendship courage',
        'animation emotions mind psychology children family growing up sadness joy',
        'animation music family death mexican culture celebration remembrance colorful',
        'crime family mafia power corruption loyalty italian american epic saga',
        'crime mafia gangster rise fall violence true story loyalty betrayal',
        'crime thriller undercover police mafia boston tension suspense double identity',
        'crime gambling mafia las vegas excess power greed downfall',
        'superhero action adventure team epic battle time travel sacrifice universe',
        'superhero action young hero responsibility power city adventure fun comic',
        'superhero action african culture technology vibranium hero villain kingdom',
        'superhero action technology genius billionaire armor invention origin story',
        'romance disaster love class ship ocean historical tragic epic emotional',
        'drama life journey history comedy love running simple man inspirational',
        'drama prison hope friendship redemption escape wrongful conviction perseverance',
        'drama historical world war two holocaust survival factory heroism sacrifice'
    ],
    'genres': [
        'Action|Crime|Thriller', 'Sci-Fi|Thriller', 'Sci-Fi|Drama', 'Mystery|Thriller',
        'Crime|Drama', 'Action|Thriller', 'Western|Action', 'War|Action',
        'Romance|Drama', 'Romance|Drama', 'Romance|Drama', 'Romance|Musical',
        'Sci-Fi|Action', 'Sci-Fi|Drama', 'Sci-Fi|Thriller', 'Romance|Sci-Fi',
        'Animation|Comedy', 'Animation|Adventure', 'Animation|Drama', 'Animation|Music',
        'Crime|Drama', 'Crime|Drama', 'Crime|Thriller', 'Crime|Drama',
        'Action|Sci-Fi', 'Action|Adventure', 'Action|Adventure', 'Action|Sci-Fi',
        'Romance|Drama', 'Drama|Comedy', 'Drama', 'Drama|History'
    ]
}

movies_df = pd.DataFrame(movies_data)

# Build the recommender
recommender = ContentBasedRecommender(max_features=1000, ngram_range=(1, 2))
recommender.fit(movies_df, text_column='description', name_column='title')

# ============================================================
# Scenario: User who loves Christopher Nolan films
# ============================================================
print("\n" + "=" * 60)
print("SCENARIO: User who loves Christopher Nolan films")
print("=" * 60)

# User has watched and rated these films
liked_movies = ['The Dark Knight', 'Inception', 'Interstellar', 'The Prestige']
liked_indices = [movies_df[movies_df['title'] == m].index[0] for m in liked_movies]
ratings = [5.0, 5.0, 4.5, 4.0]  # User ratings

print(f"\nUser's watched films (with ratings):")
for movie, rating in zip(liked_movies, ratings):
    print(f"  ★ {movie}: {rating}/5")

# Build user profile and get recommendations
user_profile = recommender.build_user_profile(liked_indices, ratings)
recs = recommender.recommend(user_profile, n=8, exclude_indices=liked_indices)

print(f"\nTop 8 Recommendations:")
print("-" * 50)
display(recs)

# ============================================================
# Find items similar to a specific movie
# ============================================================
print("\n" + "=" * 60)
print("Items similar to 'The Matrix':")
print("=" * 60)
matrix_idx = movies_df[movies_df['title'] == 'The Matrix'].index[0]
similar = recommender.find_similar_items(matrix_idx, n=5)
display(similar)

### 5.2 Naive Bayes Classifier for Content Filtering

Naive Bayes can be used as a **learning-based** content filter. Instead of computing similarities, it learns a probabilistic model of user preferences.

#### The Model

Given item features $$\mathbf{x} = (x_1, x_2, \ldots, x_d)$$, predict whether a user will like it (class $$c \in \{\text{like}, \text{dislike}\}$$):

$$P(c | \mathbf{x}) = \frac{P(\mathbf{x} | c) \cdot P(c)}{P(\mathbf{x})}$$

Using the **naive** independence assumption:

$$P(\mathbf{x} | c) = \prod_{i=1}^{d} P(x_i | c)$$

Therefore:

$$P(c | \mathbf{x}) \propto P(c) \prod_{i=1}^{d} P(x_i | c)$$

The predicted class is:

$$\hat{c} = \arg\max_{c} P(c) \prod_{i=1}^{d} P(x_i | c)$$

#### For Multinomial NB with TF-IDF features:

$$P(x_i | c) = \frac{N_{ci} + \alpha}{N_c + \alpha \cdot |V|}$$

where:
- $$N_{ci}$$ = total count of feature $$i$$ in class $$c$$
- $$N_c$$ = total count of all features in class $$c$$
- $$\alpha$$ = Laplace smoothing parameter
- $$|V|$$ = vocabulary size

#### Advantages for Content Filtering:
- Fast training and prediction
- Works well with high-dimensional sparse features (text)
- Provides probability estimates (confidence in recommendation)
- Naturally handles the "like/dislike" binary classification

**Industrial Example — Gmail Spam Filter / Priority Inbox:**  
Google's Priority Inbox uses a Naive Bayes-inspired classifier trained on email content features (sender, subject keywords, body terms) and user actions (read, reply, archive, delete) to classify incoming emails as important or not.

In [0]:
# ============================================================
# 5.2 Naive Bayes Content Filter
# Industrial Example: Email/Article recommendation classifier
# ============================================================

# Simulate a news recommendation scenario
# User has read and liked/disliked various articles
np.random.seed(42)

articles = [
    # Technology articles (user tends to like these)
    "new artificial intelligence breakthrough machine learning deep neural networks",
    "python programming tutorial data science pandas numpy advanced techniques",
    "quantum computing breakthrough processor qubits error correction",
    "cybersecurity threat ransomware attack enterprise protection strategy",
    "cloud computing aws azure kubernetes microservices deployment",
    "blockchain cryptocurrency decentralized finance smart contracts ethereum",
    "robotics autonomous vehicles self driving cars sensor fusion",
    "natural language processing transformer models bert gpt attention",
    
    # Sports articles (user tends to dislike these)
    "football championship final match goals stadium crowd celebration",
    "tennis grand slam tournament serve ace championship winner",
    "basketball playoffs scoring record dunks assists rebounds championship",
    "cricket world cup batting bowling wickets runs innings",
    "swimming olympic gold medal freestyle backstroke butterfly relay",
    "formula one racing pit stop overtake podium circuit speed",
    "boxing heavyweight championship knockout rounds belt title defense",
    "golf masters tournament birdie eagle par course green putt",
    
    # New articles to classify
    "machine learning model training optimization gradient descent convergence",
    "soccer league transfer window signings player contract negotiations",
    "virtual reality headset immersive experience gaming simulation",
    "marathon running personal best time training nutrition hydration"
]

# Labels: 1 = user likes (technology), 0 = user dislikes (sports)
labels = [1]*8 + [0]*8 + [None]*4  # Last 4 are unlabeled (to predict)

# Split into training and prediction sets
train_articles = articles[:16]
train_labels = labels[:16]
new_articles = articles[16:]
new_article_names = [
    "ML Model Optimization", "Soccer Transfers",
    "VR Gaming Experience", "Marathon Training"
]

# Vectorize
vectorizer_nb = TfidfVectorizer(max_features=500, stop_words='english')
X_train = vectorizer_nb.fit_transform(train_articles)
X_new = vectorizer_nb.transform(new_articles)
y_train = np.array(train_labels)

# Train Naive Bayes classifier
nb_classifier = MultinomialNB(alpha=1.0)  # Laplace smoothing
nb_classifier.fit(X_train, y_train)

# Predict on new articles
predictions = nb_classifier.predict(X_new)
probabilities = nb_classifier.predict_proba(X_new)

print("=" * 60)
print("Naive Bayes Content Filter - News Recommendation")
print("=" * 60)
print("\nUser Profile: Prefers TECHNOLOGY articles, dislikes SPORTS\n")

results = pd.DataFrame({
    'Article': new_article_names,
    'Prediction': ['RECOMMEND' if p == 1 else 'SKIP' for p in predictions],
    'P(Like)': probabilities[:, 1].round(4),
    'P(Dislike)': probabilities[:, 0].round(4),
    'Confidence': np.max(probabilities, axis=1).round(4)
})
display(results)

# Show most informative features
print("\nMost Informative Features:")
print("-" * 40)
feature_names = vectorizer_nb.get_feature_names_out()
log_probs_diff = nb_classifier.feature_log_prob_[1] - nb_classifier.feature_log_prob_[0]
top_like = log_probs_diff.argsort()[-10:][::-1]
top_dislike = log_probs_diff.argsort()[:10]

print("\nTop words indicating LIKE (technology):")
for idx in top_like:
    print(f"  + {feature_names[idx]}")

print("\nTop words indicating DISLIKE (sports):")
for idx in top_dislike:
    print(f"  - {feature_names[idx]}")

### 5.3 k-Nearest Neighbors (k-NN) for Content Filtering

k-NN is a **lazy learning** approach that recommends items by finding the $$k$$ most similar items in feature space to items the user has already enjoyed.

#### Algorithm

Given a query item (or user profile vector) $$\mathbf{q}$$:

1. Compute distance/similarity to every candidate item $$i$$:
   $$d(\mathbf{q}, \phi(i)) \quad \forall \, i \in C$$
2. Sort items by distance and select the $$k$$ nearest:
   $$N_k(\mathbf{q}) = \text{arg top-}k \min_i \; d(\mathbf{q}, \phi(i))$$
3. Recommend items in $$N_k(\mathbf{q})$$

For cosine-based k-NN, distance is defined as:

$$d_{\cos}(\mathbf{a}, \mathbf{b}) = 1 - \cos(\mathbf{a}, \mathbf{b})$$

#### Weighted k-NN Scoring

Rather than treating all neighbors equally, a weighted score accounts for proximity:

$$\text{score}(u, c) = \frac{\sum_{j \in N_k(c)} w_j \cdot r_{u,j}}{\sum_{j \in N_k(c)} w_j}$$

where weights decay with distance:

$$w_j = \frac{1}{d(c, j) + \epsilon}$$

or using a Gaussian kernel:

$$w_j = \exp\left(-\frac{d(c, j)^2}{2\sigma^2}\right)$$

#### Distance Metrics for k-NN

| Metric | Formula | Best For |
| --- | --- | --- |
| Cosine | $$1 - \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$$ | Text/TF-IDF |
| Euclidean | $$\sqrt{\sum(a_i - b_i)^2}$$ | Normalized numeric features |
| Manhattan | $$\sum |a_i - b_i|$$ | Sparse features |
| Minkowski | $$(\sum |a_i - b_i|^p)^{1/p}$$ | Generalized (p=1: Manhattan, p=2: Euclidean) |

#### Complexity

- **Training:** $$O(1)$$ (lazy — no training phase)
- **Inference:** $$O(n \cdot d)$$ per query (brute force), where $$n$$ = number of items, $$d$$ = dimensionality
- **Optimized:** $$O(d \cdot \log n)$$ using KD-trees or ball trees; $$O(d)$$ using approximate methods (LSH, HNSW)

#### Production Optimization: Approximate Nearest Neighbors (ANN)

For large-scale systems, exact k-NN is too slow. Production systems use:
- **Locality-Sensitive Hashing (LSH):** Hashes similar items to the same bucket
- **HNSW (Hierarchical Navigable Small World):** Graph-based approach used by FAISS, Pinecone
- **Product Quantization:** Compresses vectors for faster distance computation

**Industrial Example — Spotify Song Radio:**  
Spotify's "Song Radio" feature uses k-NN over learned audio embeddings. When a user starts a radio session from one song, the system retrieves the nearest neighbors in a 128-dimensional audio embedding space using HNSW indices, producing songs with similar acoustic properties (tempo, energy, timbre).

In [0]:
# ============================================================
# 5.3 k-Nearest Neighbors Recommender
# Industrial Example: Spotify-style music recommendation
# ============================================================

from sklearn.preprocessing import StandardScaler

# Create synthetic music dataset with audio features (Spotify-style)
music_data = {
    'song': [
        'Blinding Lights', 'Levitating', 'Save Your Tears', 'Stay',
        'Bohemian Rhapsody', 'Hotel California', 'Stairway to Heaven', 'Imagine',
        'Lose Yourself', 'HUMBLE', 'SICKO MODE', 'Gods Plan',
        'Shape of You', 'Perfect', 'Thinking Out Loud', 'Photograph',
        'Take Five', 'So What', 'Blue in Green', 'Autumn Leaves'
    ],
    'genre': [
        'pop', 'pop', 'pop', 'pop',
        'rock', 'rock', 'rock', 'rock',
        'hiphop', 'hiphop', 'hiphop', 'hiphop',
        'pop', 'pop', 'pop', 'pop',
        'jazz', 'jazz', 'jazz', 'jazz'
    ],
    # Simulated Spotify-like audio features (0-1 scale except tempo)
    'danceability': [0.85, 0.82, 0.78, 0.80, 0.39, 0.45, 0.33, 0.42,
                     0.74, 0.80, 0.71, 0.75, 0.83, 0.55, 0.60, 0.48,
                     0.52, 0.41, 0.35, 0.50],
    'energy':       [0.73, 0.76, 0.64, 0.58, 0.75, 0.67, 0.60, 0.39,
                     0.86, 0.78, 0.83, 0.65, 0.72, 0.42, 0.48, 0.35,
                     0.28, 0.22, 0.18, 0.25],
    'acousticness': [0.05, 0.08, 0.12, 0.10, 0.35, 0.42, 0.48, 0.55,
                     0.15, 0.10, 0.08, 0.20, 0.07, 0.40, 0.45, 0.62,
                     0.75, 0.82, 0.90, 0.78],
    'valence':      [0.68, 0.75, 0.58, 0.50, 0.40, 0.52, 0.38, 0.45,
                     0.72, 0.60, 0.55, 0.70, 0.82, 0.65, 0.58, 0.50,
                     0.48, 0.35, 0.30, 0.42],
    'tempo':        [171, 103, 118, 170, 72, 75, 82, 74,
                     171, 150, 155, 78, 96, 63, 79, 108,
                     174, 138, 58, 142]
}

music_df = pd.DataFrame(music_data)

# Normalize features (critical for distance-based methods)
feature_cols = ['danceability', 'energy', 'acousticness', 'valence', 'tempo']
scaler = StandardScaler()
X_music_scaled = scaler.fit_transform(music_df[feature_cols])

# Build k-NN model with cosine distance
knn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn_model.fit(X_music_scaled)

def get_song_recommendations(song_name: str, n_recommendations: int = 5):
    """Get similar songs using k-NN (Spotify Song Radio simulation)."""
    song_idx = music_df[music_df['song'] == song_name].index[0]
    song_features = X_music_scaled[song_idx].reshape(1, -1)
    
    distances, indices = knn_model.kneighbors(song_features, n_neighbors=n_recommendations + 1)
    
    recommendations = []
    for i, idx in enumerate(indices[0][1:]):  # Skip first (query song itself)
        recommendations.append({
            'rank': i + 1,
            'song': music_df.iloc[idx]['song'],
            'genre': music_df.iloc[idx]['genre'],
            'cosine_similarity': round(1 - distances[0][i + 1], 4)
        })
    return pd.DataFrame(recommendations)

# Demonstrate recommendations
print("=" * 70)
print("k-NN CONTENT-BASED MUSIC RECOMMENDER (Spotify Song Radio Style)")
print("=" * 70)

query_songs = ['Blinding Lights', 'Bohemian Rhapsody', 'Lose Yourself', 'Blue in Green']

for query in query_songs:
    print(f"\n{'─' * 60}")
    print(f"  🎵 Song Radio for: '{query}' ({music_df[music_df['song']==query]['genre'].values[0]})")
    print(f"{'─' * 60}")
    recs = get_song_recommendations(query, 4)
    display(recs)

# ============================================================
# Visualize the feature space with PCA
# ============================================================
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_music_scaled)

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
colors = {'pop': '#1DB954', 'rock': '#E74C3C', 'hiphop': '#9B59B6', 'jazz': '#F39C12'}

for genre in music_df['genre'].unique():
    mask = music_df['genre'] == genre
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
              c=colors[genre], label=genre.capitalize(), s=150, alpha=0.8, edgecolors='black')

for i, song in enumerate(music_df['song']):
    ax.annotate(song, (X_pca[i, 0] + 0.05, X_pca[i, 1] + 0.05), fontsize=7, alpha=0.85)

ax.set_title('Music Feature Space (PCA of Audio Features) \n k-NN finds nearest neighbors in this space', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(title='Genre', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 Decision Tree-Based Content Filtering

Decision trees frame content filtering as a **classification problem**: given item features, predict whether the user will like or dislike the item.

#### The Learning Framework

The tree recursively partitions the feature space by asking questions:
- Is genre == "Sci-Fi"?
- Is runtime > 120 minutes?
- Does the description contain "thriller"?

Each split is chosen to maximize **information gain** (or minimize **Gini impurity**).

#### Entropy

For a node $$S$$ with class distribution $$p_1, p_2, \ldots, p_k$$:

$$H(S) = -\sum_{i=1}^{k} p_i \log_2 p_i$$

- $$H = 0$$: perfectly pure (all samples same class)
- $$H = 1$$: maximum uncertainty (balanced binary split)

#### Information Gain

The reduction in entropy after splitting on attribute $$A$$:

$$IG(S, A) = H(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} H(S_v)$$

The attribute with the **highest information gain** is chosen at each node.

#### Gini Impurity (alternative criterion)

$$G(S) = 1 - \sum_{i=1}^{k} p_i^2$$

#### Why Trees for Content Filtering?

- **Highly interpretable:** recommendations are explainable as rules
  - "Recommended because: genre=Sci-Fi AND rating>4.0 AND has_sequel=True"
- **Handles mixed feature types** (numerical + categorical) natively
- **Captures non-linear feature interactions** automatically
- **Ensemble extensions** (Random Forest, XGBoost) dramatically improve accuracy

#### Ensemble Methods in Practice

Production systems typically use:
- **Random Forest:** reduces variance via bagging
- **Gradient Boosted Trees (XGBoost/LightGBM):** state-of-the-art for tabular features
- **Feature importance** from these models reveals what content attributes matter most

**Industrial Example — Amazon Product Recommendations:**  
Amazon uses gradient boosted tree models that learn rules like: "Users who browse 'wireless earbuds' + filter 'noise cancellation' + price range \$100–\$200 + brand preference 'Sony/Bose' are 92% likely to purchase premium audio products." The tree structure enables real-time inference on billions of candidate products.

In [0]:
# ============================================================
# 5.4 Decision Tree Content Filter
# Industrial Example: Amazon-style product recommendation
# ============================================================

np.random.seed(42)
n_products = 200

# Synthetic e-commerce product features
product_data = pd.DataFrame({
    'price': np.random.uniform(10, 500, n_products),
    'avg_rating': np.random.uniform(3.0, 5.0, n_products),
    'brand_premium': np.random.choice([0, 1], n_products, p=[0.7, 0.3]),
    'has_discount': np.random.choice([0, 1], n_products, p=[0.6, 0.4]),
    'category_electronics': np.random.choice([0, 1], n_products, p=[0.5, 0.5]),
    'review_count': np.random.randint(10, 5000, n_products),
    'free_shipping': np.random.choice([0, 1], n_products, p=[0.5, 0.5])
})

# Synthetic user preference labels (complex non-linear rules)
# User likes products that satisfy ANY of these conditions:
# 1. Electronics with good ratings (tech enthusiast)
# 2. Premium brand at moderate price (value-conscious brand loyalty)
# 3. Heavily reviewed + discounted (social proof + deal seeker)
product_data['user_likes'] = (
    ((product_data['category_electronics'] == 1) & (product_data['avg_rating'] > 4.2)) |
    ((product_data['brand_premium'] == 1) & (product_data['price'] < 200)) |
    ((product_data['has_discount'] == 1) & (product_data['review_count'] > 2000) & (product_data['free_shipping'] == 1))
).astype(int)

print(f"Dataset: {n_products} products, {product_data['user_likes'].sum()} liked by user")
print(f"Like ratio: {product_data['user_likes'].mean():.1%}\n")

# Train-test split
X = product_data.drop('user_likes', axis=1)
y = product_data['user_likes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Decision Tree
tree_model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42)
tree_model.fit(X_train, y_train)

# Evaluate
y_pred = tree_model.predict(X_test)
print("=" * 60)
print("DECISION TREE CONTENT FILTER - E-Commerce Recommendation")
print("=" * 60)
print(f"\nAccuracy:  {tree_model.score(X_test, y_test):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.3f}")

# Feature Importance Analysis
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': tree_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nLearned Feature Importance (what matters to this user):")
print("-" * 50)
display(feature_importance)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance bar chart
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis', ax=axes[0])
axes[0].set_title('Feature Importance (Decision Tree)', fontsize=12)
axes[0].set_xlabel('Gini Importance')

# Decision boundary visualization (2D slice)
from sklearn.tree import export_text
rules = export_text(tree_model, feature_names=list(X.columns), max_depth=3)
axes[1].text(0.05, 0.95, rules[:800], transform=axes[1].transAxes, fontsize=7,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1].set_title('Learned Decision Rules (Top 3 Levels)', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Predict on new products
print("\n" + "=" * 60)
print("PREDICTIONS ON NEW PRODUCTS:")
print("=" * 60)
new_products = pd.DataFrame({
    'price': [150, 350, 80, 450, 99],
    'avg_rating': [4.5, 4.8, 3.9, 4.1, 4.6],
    'brand_premium': [1, 1, 0, 1, 0],
    'has_discount': [0, 1, 1, 0, 1],
    'category_electronics': [1, 1, 0, 1, 1],
    'review_count': [2500, 500, 3000, 150, 4200],
    'free_shipping': [1, 0, 1, 1, 1]
})

new_products['prediction'] = tree_model.predict(new_products)
new_products['confidence'] = tree_model.predict_proba(new_products).max(axis=1).round(3)
new_products['decision'] = new_products['prediction'].map({1: '✓ RECOMMEND', 0: '✗ SKIP'})
display(new_products)

### 5.5 Neural Content-Based Filtering

Deep learning models learn **dense, non-linear representations** of items and users, dramatically improving over linear TF-IDF approaches.

#### Architecture Overview

A neural content-based model typically has:

1. **Item Encoder** $$f_\theta$$: Maps raw item features to a dense embedding
   $$\mathbf{v}_i = f_\theta(\text{features}_i) \in \mathbb{R}^k$$

2. **User Encoder** $$g_\phi$$: Maps user interaction history to a dense embedding
   $$\mathbf{u} = g_\phi(\{\mathbf{v}_{i_1}, \mathbf{v}_{i_2}, \ldots\}) \in \mathbb{R}^k$$

3. **Scoring Function**: Predicts affinity
   $$\hat{y}_{u,i} = \sigma(\mathbf{u}^T \mathbf{v}_i)$$ or $$\hat{y}_{u,i} = \text{MLP}([\mathbf{u}; \mathbf{v}_i])$$

#### Loss Function

For implicit feedback (clicks, views), binary cross-entropy is standard:

$$\mathcal{L} = -\sum_{(u,i) \in D} \left[ y_{u,i} \log(\hat{y}_{u,i}) + (1 - y_{u,i}) \log(1 - \hat{y}_{u,i}) \right]$$

For explicit ratings, MSE is common:

$$\mathcal{L} = \sum_{(u,i) \in D} (r_{u,i} - \hat{r}_{u,i})^2$$

#### Popular Neural Architectures

| Architecture | How It Represents Items | Use Case |
| --- | --- | --- |
| **Word2Vec/Doc2Vec** | Learns word embeddings from context | Text items |
| **CNN** | Extracts spatial features from images/spectrograms | Visual/audio items |
| **BERT/Transformers** | Contextual text embeddings | Rich text content |
| **Autoencoders** | Compresses item features to latent space | Mixed features |
| **Two-Tower Model** | Separate item & user encoders, dot-product scoring | Large-scale retrieval |

#### The Two-Tower Architecture (Google, YouTube)

The dominant architecture in production:

```
  User Tower              Item Tower
  ┌─────────┐          ┌─────────┐
  │ User    │          │ Item    │
  │ Features│          │ Features│
  └────┬────┘          └────┬────┘
       │                    │
  ┌────┴────┐          ┌────┴────┐
  │  MLP     │          │  MLP     │
  └────┬────┘          └────┬────┘
       │                    │
  ┌────┴────┐          ┌────┴────┐
  │  u ∈ R^k │─── dot ───│  v ∈ R^k │
  └─────────┘          └─────────┘
            │
       score(u, i)
```

The two towers can be independently pre-computed and cached, making inference extremely fast (a single dot product at serving time).

**Industrial Example — YouTube Recommendations:**  
YouTube's recommendation system uses a two-tower neural model. The item tower encodes video content features (title embeddings from BERT, visual embeddings from video frames, audio features, metadata). The user tower encodes watch history, search queries, and demographics. At serving time, item embeddings are pre-computed and indexed; only the user embedding needs real-time computation.

In [0]:
# ============================================================
# 5.5 Neural Content-Based Filtering
# Industrial Example: YouTube-style Two-Tower Model
# ============================================================
# Using a simplified feedforward network (no GPU dependency)

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler

# Create a richer synthetic dataset mimicking YouTube video features
np.random.seed(42)
n_items = 500

# Video content features (item tower inputs)
video_features = pd.DataFrame({
    'duration_sec': np.random.randint(30, 3600, n_items),
    'title_embedding_dim1': np.random.randn(n_items),
    'title_embedding_dim2': np.random.randn(n_items),
    'title_embedding_dim3': np.random.randn(n_items),
    'category_tech': np.random.choice([0, 1], n_items, p=[0.7, 0.3]),
    'category_music': np.random.choice([0, 1], n_items, p=[0.8, 0.2]),
    'category_gaming': np.random.choice([0, 1], n_items, p=[0.75, 0.25]),
    'avg_watch_percentage': np.random.uniform(0.2, 0.95, n_items),
    'like_ratio': np.random.uniform(0.6, 0.99, n_items),
    'upload_recency_days': np.random.randint(1, 365, n_items),
    'creator_subscriber_count': np.random.randint(1000, 10000000, n_items),
    'thumbnail_quality_score': np.random.uniform(0.3, 1.0, n_items)
})

# Complex user preference function (non-linear)
# User prefers: tech videos with high engagement, OR short music videos,
# OR highly-rated gaming content from popular creators
engagement_score = video_features['avg_watch_percentage'] * video_features['like_ratio']

user_likes = (
    ((video_features['category_tech'] == 1) & (engagement_score > 0.7)) |
    ((video_features['category_music'] == 1) & (video_features['duration_sec'] < 300)) |
    ((video_features['category_gaming'] == 1) & 
     (video_features['like_ratio'] > 0.85) & 
     (video_features['creator_subscriber_count'] > 1000000))
).astype(int)

print(f"Dataset: {n_items} videos, {user_likes.sum()} liked ({user_likes.mean():.1%})")

# Prepare data
scaler_nn = MinMaxScaler()
X_scaled = scaler_nn.fit_transform(video_features)
X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_scaled, user_likes, test_size=0.25, random_state=42, stratify=user_likes
)

# Train Neural Network (MLP = simplified "item tower" + classification head)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),  # 3-layer deep network
    activation='relu',
    learning_rate_init=0.001,
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15
)

mlp_model.fit(X_train_nn, y_train_nn)
y_pred_nn = mlp_model.predict(X_test_nn)
y_proba_nn = mlp_model.predict_proba(X_test_nn)[:, 1]

print("\n" + "=" * 60)
print("NEURAL CONTENT-BASED FILTER (Two-Tower Simplified)")
print("Industrial Parallel: YouTube Recommendation System")
print("=" * 60)
print(f"\nArchitecture: Input(12) -> Dense(128) -> Dense(64) -> Dense(32) -> Output(2)")
print(f"Activation: ReLU, Optimizer: Adam, Early Stopping: Yes")
print(f"\nTest Metrics:")
print(f"  Accuracy:  {mlp_model.score(X_test_nn, y_test_nn):.3f}")
print(f"  Precision: {precision_score(y_test_nn, y_pred_nn):.3f}")
print(f"  Recall:    {recall_score(y_test_nn, y_pred_nn):.3f}")
print(f"  F1 Score:  {f1_score(y_test_nn, y_pred_nn):.3f}")

# Compare with simpler models
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_nn, y_train_nn)
lr_pred = lr_model.predict(X_test_nn)

tree_model_2 = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model_2.fit(X_train_nn, y_train_nn)
tree_pred_2 = tree_model_2.predict(X_test_nn)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression (Linear)', 'Decision Tree (Depth=5)', 'Neural Network (3-layer MLP)'],
    'Precision': [
        precision_score(y_test_nn, lr_pred),
        precision_score(y_test_nn, tree_pred_2),
        precision_score(y_test_nn, y_pred_nn)
    ],
    'Recall': [
        recall_score(y_test_nn, lr_pred),
        recall_score(y_test_nn, tree_pred_2),
        recall_score(y_test_nn, y_pred_nn)
    ],
    'F1': [
        f1_score(y_test_nn, lr_pred),
        f1_score(y_test_nn, tree_pred_2),
        f1_score(y_test_nn, y_pred_nn)
    ]
}).round(3)

print("\nModel Comparison (same data, different architectures):")
display(comparison)

# Training loss curve
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(mlp_model.loss_curve_, label='Training Loss', color='blue', linewidth=2)
if hasattr(mlp_model, 'validation_scores_'):
    ax.plot(mlp_model.validation_scores_, label='Validation Accuracy', color='green', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss / Accuracy')
ax.set_title('Neural Content Filter: Training Convergence', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. User and Item Profile Construction

The quality of a content-based system hinges on how well user and item profiles are constructed.

### 6.1 Item Profile Construction Strategies

#### Strategy 1: Bag-of-Words / TF-IDF

Represent items by their textual content:
$$\phi_{\text{tfidf}}(i) = \text{TF-IDF}(\text{description}_i)$$

#### Strategy 2: Multi-Hot Categorical Encoding

For structured metadata (genres, tags):
$$\phi_{\text{cat}}(i) = [\mathbb{1}_{\text{action}}, \mathbb{1}_{\text{comedy}}, \mathbb{1}_{\text{drama}}, \ldots]$$

#### Strategy 3: Hybrid Profile (Concatenation)

Combine multiple feature types:
$$\phi(i) = [\alpha \cdot \phi_{\text{tfidf}}(i) \;|\; \beta \cdot \phi_{\text{cat}}(i) \;|\; \gamma \cdot \phi_{\text{num}}(i)]$$

where $$\alpha, \beta, \gamma$$ are normalization weights ensuring each feature group contributes equally.

### 6.2 User Profile Construction Strategies

#### Strategy 1: Centroid-Based (Mean Profile)

$$\mathbf{p}_u = \frac{1}{|I_u^+|} \sum_{i \in I_u^+} \phi(i)$$

Simple, fast, but dilutes strong preferences.

#### Strategy 2: Rating-Weighted Profile

$$\mathbf{p}_u = \frac{\sum_{i \in I_u} (r_{u,i} - \bar{r}_u) \cdot \phi(i)}{\sum_{i \in I_u} |r_{u,i} - \bar{r}_u|}$$

Using **mean-centered ratings** emphasizes deviations from the user's average. Items rated much higher than average contribute positively; items rated below average contribute negatively (pushing the profile away from disliked content).

#### Strategy 3: Recency-Weighted (Decay Profile)

Recent preferences matter more:

$$\mathbf{p}_u = \frac{\sum_{i \in I_u} e^{-\lambda \cdot \Delta t_i} \cdot r_{u,i} \cdot \phi(i)}{\sum_{i \in I_u} e^{-\lambda \cdot \Delta t_i} \cdot r_{u,i}}$$

where $$\Delta t_i$$ is the time since interaction with item $$i$$, and $$\lambda$$ controls the decay rate.

#### Strategy 4: Rocchio Algorithm (from Information Retrieval)

The Rocchio method explicitly models both positive and negative feedback:

$$\mathbf{p}_u = \alpha \cdot \mathbf{p}_u^{(0)} + \beta \cdot \frac{1}{|I_u^+|} \sum_{i \in I_u^+} \phi(i) - \gamma \cdot \frac{1}{|I_u^-|} \sum_{i \in I_u^-} \phi(i)$$

where:
- $$\mathbf{p}_u^{(0)}$$ = initial query/profile
- $$I_u^+$$ = positively rated items
- $$I_u^-$$ = negatively rated items
- $$\alpha, \beta, \gamma$$ = weights for original query, positive, and negative feedback

Typical values: $$\alpha = 1, \beta = 0.75, \gamma = 0.25$$

### 6.3 Cold-Start Handling

When a new user has no history:
- Use **popularity-based** defaults
- Employ **active learning**: present diverse items and ask for ratings
- Use **demographic priors**: initialize profile from similar user demographics

In [0]:
# ============================================================
# 6. User Profile Construction - All Strategies Demonstrated
# ============================================================

# Reuse the movie recommender from Section 5.1
# We'll show how different profile strategies change recommendations

print("=" * 70)
print("USER PROFILE CONSTRUCTION STRATEGIES COMPARISON")
print("=" * 70)

# Simulate a user with varied ratings over time
user_movies = ['The Dark Knight', 'Inception', 'Pulp Fiction', 'The Notebook', 'Interstellar']
user_ratings = [5.0, 5.0, 4.0, 2.0, 4.5]  # Note: 'The Notebook' rated low
user_timestamps = [365, 300, 200, 150, 30]  # Days ago (Interstellar most recent)

user_indices = [movies_df[movies_df['title'] == m].index[0] for m in user_movies]

print("\nUser's Rating History:")
print("-" * 50)
for movie, rating, days in zip(user_movies, user_ratings, user_timestamps):
    print(f"  {movie:<25} Rating: {rating}/5  ({days} days ago)")

# Strategy 1: Simple Mean
print("\n" + "~" * 70)
print("Strategy 1: SIMPLE MEAN (all items weighted equally)")
print("~" * 70)
mean_profile = recommender.build_user_profile(user_indices)
mean_recs = recommender.recommend(mean_profile, n=5, exclude_indices=user_indices)
display(mean_recs)

# Strategy 2: Rating-Weighted
print("\n" + "~" * 70)
print("Strategy 2: RATING-WEIGHTED (higher-rated items contribute more)")
print("~" * 70)
weighted_profile = recommender.build_user_profile(user_indices, ratings=user_ratings)
weighted_recs = recommender.recommend(weighted_profile, n=5, exclude_indices=user_indices)
display(weighted_recs)

# Strategy 3: Mean-Centered Rating-Weighted
print("\n" + "~" * 70)
print("Strategy 3: MEAN-CENTERED (negative contributions from disliked items)")
print("~" * 70)
mean_rating = np.mean(user_ratings)
centered_ratings = [r - mean_rating for r in user_ratings]
# Only use positive-centered weights for profile (negative pushes away)
pos_indices = [idx for idx, r in zip(user_indices, centered_ratings) if r > 0]
pos_weights = [r for r in centered_ratings if r > 0]
centered_profile = recommender.build_user_profile(pos_indices, ratings=pos_weights)
centered_recs = recommender.recommend(centered_profile, n=5, exclude_indices=user_indices)
display(centered_recs)

# Strategy 4: Recency-Weighted (exponential decay)
print("\n" + "~" * 70)
print("Strategy 4: RECENCY-WEIGHTED (recent interactions matter more, λ=0.01)")
print("~" * 70)
lambda_decay = 0.01
recency_weights = [rating * np.exp(-lambda_decay * days) 
                   for rating, days in zip(user_ratings, user_timestamps)]
recency_profile = recommender.build_user_profile(user_indices, ratings=recency_weights)
recency_recs = recommender.recommend(recency_profile, n=5, exclude_indices=user_indices)
display(recency_recs)

# Summary comparison
print("\n" + "=" * 70)
print("COMPARISON: Top recommendation from each strategy")
print("=" * 70)
comparison_df = pd.DataFrame({
    'Strategy': ['Simple Mean', 'Rating-Weighted', 'Mean-Centered', 'Recency-Weighted'],
    'Top Recommendation': [
        mean_recs.iloc[0]['item'],
        weighted_recs.iloc[0]['item'],
        centered_recs.iloc[0]['item'],
        recency_recs.iloc[0]['item']
    ],
    'Score': [
        mean_recs.iloc[0]['similarity_score'],
        weighted_recs.iloc[0]['similarity_score'],
        centered_recs.iloc[0]['similarity_score'],
        recency_recs.iloc[0]['similarity_score']
    ]
})
comparison_df['Score'] = comparison_df['Score'].round(4)
display(comparison_df)

## 7. Industrial Examples (Deep Dive)

### 7.1 Netflix — Movie & TV Recommendations

**Scale:** 230M+ subscribers, 17,000+ titles, billions of viewing events daily

**Content Features Used:**
- **Structured:** Genre tags, cast, director, country, release year, runtime, maturity rating
- **Textual:** Plot synopses, user-generated tags, critic reviews
- **Visual:** Key frame analysis (color palette, scene composition) for thumbnail personalization
- **Behavioral:** Viewing completion rate, re-watch signals, time-of-day patterns

**Architecture:**
- Content-based component uses deep autoencoders on video metadata
- Combined with collaborative filtering in a hybrid model
- A/B tested extensively (Netflix famously uses 100+ simultaneous experiments)

**Key Insight:** Netflix found that combining genre tags with plot keywords improved content-based recommendations by 15% over genre-only approaches.

---

### 7.2 Spotify — Music Discovery

**Scale:** 600M+ users, 100M+ tracks, 5B+ playlists

**Content Features Used:**
- **Audio:** Mel-frequency cepstral coefficients (MFCCs), tempo, key, loudness, energy, danceability, instrumentalness, speechiness (extracted by CNN on spectrograms)
- **Text:** Artist bios, song lyrics, playlist titles, music blog text (NLP embeddings)
- **Cultural:** Playlist co-occurrence (if two songs appear in many playlists together, they're related)

**Systems:**
- **Discover Weekly:** Hybrid of collaborative filtering + content-based (audio CNN embeddings)
- **Release Radar:** Content-based matching of new releases to user taste profiles
- **Song Radio:** k-NN over audio embedding space

**Key Innovation:** Spotify's deep content analysis solves the cold-start problem for new artists with zero listeners — the audio CNN can embed a brand-new track and find similar known tracks purely from sound.

---

### 7.3 Google News — Article Recommendations

**Scale:** Billions of articles, hundreds of millions of daily users

**Content Features Used:**
- **Text:** Article body TF-IDF, named entities (people, organizations, locations), topic models (LDA)
- **Source:** Publisher credibility, editorial quality signals
- **Temporal:** Recency decay (news is highly time-sensitive)
- **Structural:** Article length, multimedia presence, headline patterns

**Architecture:**
- NRMS (Neural News Recommendation with Multi-Head Self-Attention)
- Uses BERT-based encoders for news articles
- User model: multi-head attention over clicked article embeddings

**Key Challenge:** News has extreme cold-start (most articles are read within hours of publication). Content-based approaches are critical here because collaborative signals haven't accumulated yet for new articles.

---

### 7.4 Amazon — Product Recommendations

**Scale:** 350M+ products, 300M+ customers

**Content Features Used:**
- **Structured:** Category taxonomy (6+ levels deep), brand, price, specifications, dimensions
- **Text:** Product titles, bullet points, descriptions, Q&A, reviews
- **Visual:** Product image embeddings (CNN-based)
- **Behavioral:** Browse patterns, add-to-cart without purchase, search queries

**Architecture:**
- Item-to-item content similarity ("Customers who viewed this also viewed...")
- Feature-based personalized ranking (LambdaMART with content features)
- Deep learning models for cross-category recommendations

**Key Innovation:** Amazon's "Frequently Bought Together" combines content similarity (complementary products) with purchase co-occurrence data.

In [0]:
# ============================================================
# 7.3 Industrial Example: Google News-style Article Recommender
# Demonstrates: TF-IDF + Cosine + Recency Decay
# ============================================================

from datetime import datetime, timedelta

# Simulate a news corpus
news_articles = {
    'title': [
        'AI Breakthrough: New Language Model Surpasses Human Benchmarks',
        'Federal Reserve Raises Interest Rates by 0.25 Percent',
        'SpaceX Successfully Launches Starship to Mars Orbit',
        'Climate Summit: Nations Agree to Carbon Reduction Targets',
        'Tech Giants Report Record Quarterly Earnings',
        'New Study Links Exercise to Improved Mental Health',
        'Quantum Computer Achieves Error Correction Milestone',
        'Housing Market Cools as Mortgage Rates Hit Decade High',
        'CRISPR Gene Therapy Shows Promise in Clinical Trials',
        'Electric Vehicle Sales Surpass Gas Cars in Europe',
        'Cybersecurity Threat: Major Data Breach Affects Millions',
        'Olympic Committee Announces New Sports for 2028 Games',
        'Artificial Intelligence Regulation Framework Proposed by EU',
        'Stock Market Volatile After Inflation Data Release',
        'NASA Discovers Water Ice on Lunar South Pole',
        'Renewable Energy Investment Hits Record 500 Billion',
        'Deep Learning Model Predicts Protein Structures Accurately',
        'Central Bank Digital Currency Pilot Launches in Asia',
        'Autonomous Vehicles Approved for Highway Testing',
        'Global Chip Shortage Eases as New Fabs Come Online'
    ],
    'content': [
        'artificial intelligence machine learning transformer neural network nlp benchmark performance gpt language model deep learning',
        'federal reserve interest rates monetary policy inflation economy banking central bank yield bond market',
        'spacex starship mars rocket launch orbit payload reusable spacecraft elon musk aerospace',
        'climate change carbon emissions global warming paris agreement sustainability renewable cop summit nations',
        'technology companies apple google microsoft amazon meta earnings revenue profit quarterly growth',
        'exercise physical activity mental health depression anxiety wellness study research brain fitness',
        'quantum computing qubit error correction supremacy algorithm processor silicon entanglement breakthrough',
        'housing market mortgage rates real estate prices homes buyers sellers affordability supply demand',
        'crispr gene editing therapy clinical trials patients dna genetic disease treatment cure biology',
        'electric vehicles ev tesla charging battery range sales europe market adoption emission',
        'cybersecurity data breach hack vulnerability encryption ransomware attack protection enterprise security',
        'olympics sports athletes competition games events committee international medal ceremony',
        'artificial intelligence regulation policy governance ethics framework compliance european union law technology',
        'stock market volatility inflation data consumer prices trading investors dow nasdaq sp500 correction',
        'nasa space exploration moon lunar water ice south pole artemis mission discovery',
        'renewable energy solar wind investment clean power capacity gigawatt installation record growth',
        'deep learning protein structure prediction alphafold biology amino acid folding molecular model',
        'digital currency central bank cbdc blockchain payments financial system pilot money transaction',
        'autonomous vehicles self driving cars highway testing safety sensors lidar regulation approval',
        'semiconductor chip manufacturing fab supply shortage production wafer technology intel tsmc'
    ],
    'category': [
        'Tech', 'Finance', 'Space', 'Environment', 'Tech',
        'Health', 'Tech', 'Finance', 'Health', 'Auto',
        'Tech', 'Sports', 'Tech', 'Finance', 'Space',
        'Energy', 'Tech', 'Finance', 'Auto', 'Tech'
    ],
    'hours_ago': [
        2, 5, 1, 24, 8, 48, 3, 12, 36, 6,
        4, 72, 7, 10, 18, 30, 5, 15, 9, 20
    ]
}

news_df = pd.DataFrame(news_articles)

# Build TF-IDF representation
news_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
news_tfidf = news_vectorizer.fit_transform(news_df['content'])

# Simulate user reading history (user likes Tech & Space)
user_read_indices = [0, 2, 6, 10]  # AI Breakthrough, SpaceX, Quantum, Cybersecurity
user_read_titles = news_df.iloc[user_read_indices]['title'].tolist()

print("=" * 70)
print("GOOGLE NEWS-STYLE CONTENT RECOMMENDER")
print("=" * 70)
print("\nUser's Reading History:")
for idx in user_read_indices:
    print(f"  • [{news_df.iloc[idx]['category']}] {news_df.iloc[idx]['title']}")

# Build user profile (mean of read articles)
user_profile_news = news_tfidf[user_read_indices].mean(axis=0)
user_profile_news = np.asarray(user_profile_news)

# Compute content similarity
content_scores = cosine_similarity(user_profile_news, news_tfidf).flatten()

# Apply recency decay (news freshness matters!)
lambda_recency = 0.05  # Decay rate per hour
recency_weights = np.exp(-lambda_recency * news_df['hours_ago'].values)

# Final score = content_similarity * recency_weight
final_scores = content_scores * recency_weights

# Exclude already-read articles
for idx in user_read_indices:
    final_scores[idx] = -1

# Get top recommendations
top_indices = final_scores.argsort()[-8:][::-1]

results = pd.DataFrame({
    'Rank': range(1, 9),
    'Article': news_df.iloc[top_indices]['title'].values,
    'Category': news_df.iloc[top_indices]['category'].values,
    'Content Score': content_scores[top_indices].round(4),
    'Recency Weight': recency_weights[top_indices].round(4),
    'Final Score': final_scores[top_indices].round(4),
    'Hours Ago': news_df.iloc[top_indices]['hours_ago'].values
})

print("\nTop 8 Recommended Articles (Content Similarity × Recency):")
print("-" * 70)
display(results)

# Visualization: Content score vs Recency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Score decomposition for top articles
x_pos = range(len(results))
axes[0].barh(x_pos, results['Content Score'], color='steelblue', alpha=0.7, label='Content Score')
axes[0].barh(x_pos, results['Recency Weight'], left=results['Content Score'],
             color='coral', alpha=0.7, label='Recency Boost')
axes[0].set_yticks(x_pos)
axes[0].set_yticklabels([t[:35] + '...' if len(t) > 35 else t for t in results['Article']], fontsize=8)
axes[0].set_xlabel('Score Components')
axes[0].set_title('Score Decomposition: Content vs Recency')
axes[0].legend()

# Right: All articles in content-score vs freshness space
scatter_colors = {'Tech': 'blue', 'Finance': 'green', 'Space': 'red', 
                  'Health': 'purple', 'Auto': 'orange', 'Energy': 'brown',
                  'Sports': 'gray', 'Environment': 'teal'}
for cat in news_df['category'].unique():
    mask = news_df['category'] == cat
    axes[1].scatter(
        news_df[mask]['hours_ago'], content_scores[mask],
        c=scatter_colors.get(cat, 'black'), label=cat, s=80, alpha=0.7
    )
axes[1].set_xlabel('Hours Since Published')
axes[1].set_ylabel('Content Similarity to User')
axes[1].set_title('All Articles: Relevance vs Freshness')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluation Metrics for Content-Based Systems

Evaluating recommender systems requires metrics that capture both **accuracy** and **utility** of recommendations.

### 8.1 Classification Metrics (Binary Relevance)

When framing recommendation as "will the user engage?"

#### Precision@k

Fraction of recommended items that are relevant:

$$\text{Precision@}k = \frac{|\{\text{relevant items}\} \cap \{\text{top-}k \text{ items}\}|}{k}$$

#### Recall@k

Fraction of relevant items that appear in top-k:

$$\text{Recall@}k = \frac{|\{\text{relevant items}\} \cap \{\text{top-}k \text{ items}\}|}{|\{\text{relevant items}\}|}$$

#### F1@k

Harmonic mean balancing precision and recall:

$$F1@k = 2 \cdot \frac{\text{Precision@}k \cdot \text{Recall@}k}{\text{Precision@}k + \text{Recall@}k}$$

### 8.2 Ranking Metrics

#### Mean Average Precision (MAP)

Averages precision at each relevant item's position:

$$\text{AP} = \frac{1}{|\text{rel}|} \sum_{k=1}^{n} \text{Precision@}k \cdot \text{rel}(k)$$

$$\text{MAP} = \frac{1}{|U|} \sum_{u \in U} \text{AP}(u)$$

#### Normalized Discounted Cumulative Gain (NDCG@k)

Accounts for graded relevance and position discount:

$$\text{DCG@}k = \sum_{i=1}^{k} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}$$

$$\text{NDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}$$

where IDCG is the ideal (best possible) DCG.

#### Mean Reciprocal Rank (MRR)

Position of the first relevant item:

$$\text{MRR} = \frac{1}{|U|} \sum_{u=1}^{|U|} \frac{1}{\text{rank}_u}$$

### 8.3 Beyond-Accuracy Metrics

| Metric | Measures | Formula |
| --- | --- | --- |
| **Coverage** | Fraction of catalog recommended | $$\frac{|\bigcup_u \text{Rec}(u)|}{|I|}$$ |
| **Diversity** | How different recommendations are from each other | $$1 - \frac{2}{k(k-1)} \sum_{i<j} \text{sim}(r_i, r_j)$$ |
| **Novelty** | How surprising/unknown items are | $$\frac{1}{k} \sum_{i=1}^{k} -\log_2 p(i)$$ |
| **Serendipity** | Unexpected yet relevant | Coverage of items outside user's filter bubble |

### 8.4 The Filter Bubble Problem

Content-based filtering tends to create **filter bubbles** — users only see items similar to what they've already consumed. Metrics like diversity and serendipity help detect and mitigate this.

In [0]:
# ============================================================
# 8. Evaluation Metrics - Complete Implementation
# ============================================================

def precision_at_k(recommended: list, relevant: set, k: int) -> float:
    """Precision@k: fraction of top-k that are relevant."""
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & relevant)
    return hits / k

def recall_at_k(recommended: list, relevant: set, k: int) -> float:
    """Recall@k: fraction of relevant items found in top-k."""
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & relevant)
    return hits / len(relevant) if relevant else 0.0

def average_precision(recommended: list, relevant: set) -> float:
    """Average Precision for a single user."""
    hits = 0
    sum_precision = 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            hits += 1
            sum_precision += hits / (i + 1)
    return sum_precision / len(relevant) if relevant else 0.0

def ndcg_at_k(recommended: list, relevant: dict, k: int) -> float:
    """
    NDCG@k with graded relevance.
    relevant: dict mapping item -> relevance score
    """
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        rel = relevant.get(item, 0)
        dcg += (2**rel - 1) / np.log2(i + 2)  # i+2 because log2(1) = 0
    
    # Ideal DCG
    ideal_rels = sorted(relevant.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_rels))
    
    return dcg / idcg if idcg > 0 else 0.0

def diversity(recommended: list, similarity_matrix: np.ndarray, item_index_map: dict) -> float:
    """Intra-list diversity: 1 - avg pairwise similarity."""
    k = len(recommended)
    if k < 2:
        return 0.0
    
    total_sim = 0.0
    pairs = 0
    for i in range(k):
        for j in range(i + 1, k):
            idx_i = item_index_map.get(recommended[i], 0)
            idx_j = item_index_map.get(recommended[j], 0)
            total_sim += similarity_matrix[idx_i, idx_j]
            pairs += 1
    
    avg_sim = total_sim / pairs
    return 1 - avg_sim

def coverage(all_recommendations: list, total_items: int) -> float:
    """Catalog coverage: fraction of items ever recommended."""
    unique_recommended = set(item for rec_list in all_recommendations for item in rec_list)
    return len(unique_recommended) / total_items


# ============================================================
# Evaluate our Netflix-style recommender
# ============================================================
print("=" * 70)
print("EVALUATION METRICS DEMONSTRATION")
print("=" * 70)

# Simulated ground truth: items the user would actually like
# (imagine we know from a held-out test set)
true_relevant = {'The Matrix', 'Blade Runner 2049', 'Ex Machina', 'Her', 
                 'The Prestige', 'The Departed', 'Goodfellas'}

# Our system's recommendations (from the Nolan fan scenario)
recommended_items = list(weighted_recs['item'].values)

# Graded relevance (for NDCG)
graded_relevance = {
    'The Matrix': 5, 'Blade Runner 2049': 4, 'Ex Machina': 5,
    'Her': 3, 'The Prestige': 4, 'The Departed': 3, 'Goodfellas': 2
}

print(f"\nGround truth relevant items: {true_relevant}")
print(f"System recommendations: {recommended_items}")

# Compute metrics at different k values
k_values = [1, 3, 5, 8]
metrics_table = []

for k in k_values:
    p = precision_at_k(recommended_items, true_relevant, k)
    r = recall_at_k(recommended_items, true_relevant, k)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    ndcg = ndcg_at_k(recommended_items, graded_relevance, k)
    
    metrics_table.append({
        'k': k,
        'Precision@k': round(p, 4),
        'Recall@k': round(r, 4),
        'F1@k': round(f1, 4),
        'NDCG@k': round(ndcg, 4)
    })

metrics_df = pd.DataFrame(metrics_table)
print("\nMetrics at various k:")
display(metrics_df)

# Average Precision
ap = average_precision(recommended_items, true_relevant)
print(f"\nAverage Precision (AP): {ap:.4f}")

# Diversity (using our similarity matrix from earlier)
item_idx_map = {title: i for i, title in enumerate(movies_df['title'])}
rec_indices = [item_idx_map[item] for item in recommended_items if item in item_idx_map]
if len(rec_indices) >= 2:
    rec_sim_matrix = cosine_similarity(
        recommender.item_profiles[rec_indices]
    )
    avg_pairwise_sim = (rec_sim_matrix.sum() - np.trace(rec_sim_matrix)) / (len(rec_indices) * (len(rec_indices) - 1))
    div = 1 - avg_pairwise_sim
    print(f"Intra-list Diversity: {div:.4f} (1.0 = perfectly diverse, 0.0 = all identical)")

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
bar_width = 0.2
x = np.arange(len(k_values))

ax.bar(x - 1.5*bar_width, metrics_df['Precision@k'], bar_width, label='Precision@k', color='#2ecc71')
ax.bar(x - 0.5*bar_width, metrics_df['Recall@k'], bar_width, label='Recall@k', color='#3498db')
ax.bar(x + 0.5*bar_width, metrics_df['F1@k'], bar_width, label='F1@k', color='#e74c3c')
ax.bar(x + 1.5*bar_width, metrics_df['NDCG@k'], bar_width, label='NDCG@k', color='#9b59b6')

ax.set_xlabel('k')
ax.set_ylabel('Score')
ax.set_title('Recommendation Quality Metrics at Various k', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([f'k={k}' for k in k_values])
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Advantages, Limitations, and Hybrid Approaches

### 9.1 Advantages of Content-Based Filtering

| Advantage | Explanation |
| --- | --- |
| **No cold-start for new items** | Items can be recommended immediately based on their features — no user interactions required |
| **User independence** | Recommendations depend only on the individual user's profile, not on other users |
| **Transparency / Explainability** | Can explain WHY an item was recommended ("Because you liked sci-fi movies with time travel") |
| **No popularity bias** | Niche items with unique content features can be recommended even without many ratings |
| **Privacy-preserving** | No need to share user behavior data across users |
| **Works with sparse data** | Doesn't need dense user-item interaction matrices |

### 9.2 Limitations

| Limitation | Explanation | Mitigation |
| --- | --- | --- |
| **Filter bubble** | Over-specialization — only recommends more of the same | Add diversity constraints, exploration |
| **Feature engineering** | Quality depends heavily on feature quality and availability | Use deep learning for automatic feature extraction |
| **Cannot capture quality** | Two items may have identical features but vastly different quality | Incorporate rating/engagement signals |
| **Cross-domain limitations** | Cannot recommend items in new domains user hasn't explored | Use knowledge graphs for cross-domain transfer |
| **Cold-start for new users** | Need some initial preferences to build a profile | Active learning, demographic priors |
| **Serendipity** | Rarely surfaces surprising, unexpected recommendations | Hybrid with collaborative filtering |

### 9.3 Hybrid Approaches

Most production systems combine content-based and collaborative filtering:

#### Weighted Hybrid

$$\text{score}_{\text{hybrid}}(u, i) = \alpha \cdot \text{score}_{\text{CB}}(u, i) + (1 - \alpha) \cdot \text{score}_{\text{CF}}(u, i)$$

#### Feature Augmentation

Use content-based features as additional inputs to a collaborative model:
$$\hat{r}_{u,i} = \mathbf{p}_u^T \mathbf{q}_i + \mathbf{w}^T \phi(i)$$

where $$\mathbf{p}_u, \mathbf{q}_i$$ are latent factors (CF) and $$\phi(i)$$ are content features.

#### Cascade Hybrid

1. **Stage 1 (Retrieval):** Content-based filtering generates candidates (fast, broad)
2. **Stage 2 (Ranking):** Collaborative/neural model re-ranks candidates (accurate, personalized)

This is the architecture used by YouTube, Netflix, and most large-scale systems.

#### Meta-Level Hybrid

The content-based model generates a **representation** that is used as input to the collaborative model (or vice versa).

### 9.4 When to Use Content-Based Filtering

**Prefer content-based when:**
- New items appear frequently (news, job postings, products)
- Rich item metadata is available
- User privacy is paramount
- Explainability is required (regulated domains)
- The user base is small or sparse

**Prefer collaborative when:**
- Item features are hard to extract (e.g., jokes, memes)
- Discovery/serendipity is important
- Abundant user-item interaction data exists
- Quality/taste signals matter more than content attributes

In [0]:
# ============================================================
# 9.3 Hybrid Approach: Combining Content + Popularity (Simple Demo)
# ============================================================

# Demonstrate how hybrid scoring improves over pure content-based

# Simulate popularity scores (global engagement signals)
np.random.seed(123)
movies_df['popularity'] = np.random.uniform(0.2, 1.0, len(movies_df))

# Boost popular items slightly
popularity_boost = {
    'The Dark Knight': 0.95, 'Inception': 0.92, 'Interstellar': 0.88,
    'Pulp Fiction': 0.90, 'The Godfather': 0.97, 'The Matrix': 0.85,
    'Avengers Endgame': 0.93, 'Toy Story': 0.87, 'Forrest Gump': 0.91,
    'Titanic': 0.94
}
for title, pop in popularity_boost.items():
    idx = movies_df[movies_df['title'] == title].index[0]
    movies_df.loc[idx, 'popularity'] = pop

# Get content-based scores from our recommender (Nolan fan scenario)
content_scores_all = cosine_similarity(weighted_profile, recommender.item_profiles).flatten()

# Hybrid: alpha * content + (1-alpha) * popularity
alphas = [1.0, 0.8, 0.6, 0.5]

print("=" * 70)
print("HYBRID RECOMMENDER: Content-Based + Popularity")
print("=" * 70)
print("\nFormula: score = α × content_similarity + (1-α) × popularity")
print("\nUser Profile: Nolan film enthusiast")

for alpha in alphas:
    hybrid_scores = alpha * content_scores_all + (1 - alpha) * movies_df['popularity'].values
    
    # Exclude watched movies
    for idx in liked_indices:
        hybrid_scores[idx] = -1
    
    top5 = hybrid_scores.argsort()[-5:][::-1]
    
    print(f"\n{"─" * 50}")
    print(f"  α = {alpha} ({'Pure Content' if alpha == 1.0 else f'{int(alpha*100)}% Content + {int((1-alpha)*100)}% Popularity'})")
    print(f"{"─" * 50}")
    for rank, idx in enumerate(top5, 1):
        print(f"  {rank}. {movies_df.iloc[idx]['title']:<30} "
              f"(content={content_scores_all[idx]:.3f}, "
              f"pop={movies_df.iloc[idx]['popularity']:.3f}, "
              f"hybrid={hybrid_scores[idx]:.3f})")

print("\n" + "=" * 70)
print("OBSERVATION: Lower α introduces popular items that are less content-similar")
print("but globally well-received — increasing serendipity and reducing filter bubble.")
print("=" * 70)

## Appendix A: Embedding-Based Content Filtering — Word2Vec and Beyond

### From Sparse to Dense: Why Embeddings?

TF-IDF vectors are **sparse** and **high-dimensional** (vocabulary size can be 50k+). This causes:
- High memory usage
- Inability to capture semantic similarity ("car" and "automobile" are orthogonal in TF-IDF)
- Curse of dimensionality for k-NN

**Embeddings** learn **dense, low-dimensional** representations (typically 64–768 dimensions) where semantically similar concepts are close together.

### Word2Vec (Mikolov et al., 2013)

Learns word embeddings from context windows using two architectures:

#### Skip-Gram Model

Given a target word $$w_t$$, predict surrounding context words $$w_{t-c}, \ldots, w_{t+c}$$:

$$\max \sum_{t=1}^{T} \sum_{-c \leq j \leq c, j \neq 0} \log P(w_{t+j} | w_t)$$

where:

$$P(w_O | w_I) = \frac{\exp(\mathbf{v}_{w_O}' \cdot \mathbf{v}_{w_I})}{\sum_{w=1}^{W} \exp(\mathbf{v}_w' \cdot \mathbf{v}_{w_I})}$$

#### CBOW (Continuous Bag of Words)

Predicts the target word from its context (reverse of Skip-Gram):

$$P(w_t | w_{t-c}, \ldots, w_{t+c}) = \text{softmax}(\mathbf{W} \cdot \frac{1}{2c} \sum_{j} \mathbf{v}_{w_{t+j}})$$

### Doc2Vec / Paragraph Vectors (Le & Mikolov, 2014)

Extends Word2Vec to learn embeddings for entire documents. Each document gets a unique vector that participates in the prediction task alongside word vectors.

### BERT Embeddings (Devlin et al., 2019)

Transformer-based contextual embeddings that capture word meaning in context:
- "bank" near "river" → different embedding than "bank" near "money"
- State-of-the-art for text similarity tasks
- Pre-trained on massive corpora, fine-tuned for specific domains

### Item Embedding from Embeddings

Given word embeddings for all words in an item's description, the item embedding can be:

$$\mathbf{v}_i = \frac{1}{|d_i|} \sum_{w \in d_i} \mathbf{e}_w \quad \text{(mean pooling)}$$

or using TF-IDF weighted averaging:

$$\mathbf{v}_i = \frac{\sum_{w \in d_i} \text{tfidf}(w, d_i) \cdot \mathbf{e}_w}{\sum_{w \in d_i} \text{tfidf}(w, d_i)}$$

In [0]:
# ============================================================
# Appendix A: Word Embedding-Based Content Filtering
# Using sklearn's truncated SVD as a lightweight embedding proxy
# (In production: Word2Vec, BERT, or Sentence-BERT)
# ============================================================

from sklearn.decomposition import TruncatedSVD

# TF-IDF -> SVD = Latent Semantic Analysis (LSA)
# This approximates the effect of dense embeddings

# Use our movie corpus TF-IDF matrix (from the recommender)
print("=" * 70)
print("EMBEDDING-BASED CONTENT FILTERING (LSA / Truncated SVD)")
print("=" * 70)
print("\nTF-IDF matrix shape:", recommender.item_profiles.shape)

# Reduce to dense 10-dimensional embeddings (simulating Word2Vec/BERT)
n_components = 10
svd = TruncatedSVD(n_components=n_components, random_state=42)
item_embeddings = svd.fit_transform(recommender.item_profiles)

print(f"Dense embedding shape: {item_embeddings.shape}")
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.1%}")
print(f"\nDimensionality reduction: {recommender.item_profiles.shape[1]} -> {n_components} dims")

# Compare similarity rankings: sparse TF-IDF vs dense embeddings
query_movie = 'Inception'
query_idx = movies_df[movies_df['title'] == query_movie].index[0]

# Sparse similarities (TF-IDF cosine)
sparse_sims = cosine_similarity(
    recommender.item_profiles[query_idx], recommender.item_profiles
).flatten()
sparse_sims[query_idx] = -1

# Dense similarities (embedding cosine)
dense_sims = cosine_similarity(
    item_embeddings[query_idx].reshape(1, -1), item_embeddings
).flatten()
dense_sims[query_idx] = -1

# Top 5 from each
sparse_top5 = sparse_sims.argsort()[-5:][::-1]
dense_top5 = dense_sims.argsort()[-5:][::-1]

print(f"\n{'=' * 60}")
print(f"Similar to '{query_movie}' - Sparse vs Dense Representations")
print(f"{'=' * 60}")

comp_df = pd.DataFrame({
    'Rank': range(1, 6),
    'TF-IDF (Sparse, d=1000)': [f"{movies_df.iloc[i]['title']} ({sparse_sims[i]:.3f})" for i in sparse_top5],
    'LSA Embedding (Dense, d=10)': [f"{movies_df.iloc[i]['title']} ({dense_sims[i]:.3f})" for i in dense_top5]
})
display(comp_df)

# Visualize embedding space (2D)
pca_emb = PCA(n_components=2)
emb_2d = pca_emb.fit_transform(item_embeddings)

fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Color by genre (primary)
genre_colors = {
    'Action': '#E74C3C', 'Sci-Fi': '#3498DB', 'Romance': '#E91E63',
    'Crime': '#2C3E50', 'Animation': '#27AE60', 'Drama': '#8E44AD',
    'Mystery': '#F39C12', 'Western': '#D35400', 'War': '#1ABC9C'
}

for i, row in movies_df.iterrows():
    primary_genre = row['genres'].split('|')[0]
    color = genre_colors.get(primary_genre, '#95A5A6')
    ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c=color, s=100, alpha=0.8, edgecolors='black', linewidth=0.5)
    ax.annotate(row['title'], (emb_2d[i, 0] + 0.02, emb_2d[i, 1] + 0.02), fontsize=7)

# Legend
for genre, color in genre_colors.items():
    ax.scatter([], [], c=color, s=80, label=genre)
ax.legend(title='Primary Genre', loc='best', fontsize=9)
ax.set_title('Movie Embedding Space (LSA: TF-IDF → 10D → 2D PCA)', fontsize=13)
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("\nKey Insight: In embedding space, semantically similar movies cluster together")
print("even when they don't share exact words (e.g., 'Ex Machina' and 'Her' both")
print("involve AI/consciousness themes despite different vocabulary).")

## Appendix B: Scalability Considerations in Production

### The Two-Stage Architecture

All large-scale recommendation systems follow a **retrieval + ranking** pipeline:

```
  Billions of items
       │
  ┌────┴──────────────┐
  │  RETRIEVAL (fast)   │  ← Content-based k-NN, ANN indices
  │  O(log n) per query │     (FAISS, ScaNN, Pinecone)
  └────┬──────────────┘
       │
  ~1000 candidates
       │
  ┌────┴──────────────┐
  │  RANKING (accurate)  │  ← Neural models, cross-features
  │  O(1000 × model)    │     (transformers, deep & cross)
  └────┬──────────────┘
       │
  ~10-50 shown to user
```

### Approximate Nearest Neighbor (ANN) Methods

| Method | Mechanism | Index Build | Query Time | Used By |
| --- | --- | --- | --- | --- |
| **LSH** | Hash similar vectors to same bucket | $$O(n)$$ | $$O(1)$$ expected | Early systems |
| **HNSW** | Hierarchical graph navigation | $$O(n \log n)$$ | $$O(\log n)$$ | FAISS, Pinecone |
| **IVF** | Cluster + search nearby clusters | $$O(n)$$ | $$O(\sqrt{n})$$ | FAISS |
| **Product Quantization** | Compress vectors to codes | $$O(n)$$ | $$O(n/\text{compression})$$ | FAISS, ScaNN |

### Serving Architecture

- **Offline:** Pre-compute all item embeddings, build ANN index
- **Online:** Compute user embedding in real-time, query ANN index
- **Latency budget:** Typically 50–200ms for the full retrieval + ranking pipeline
- **Refresh rate:** Item index rebuilt hourly/daily; user profiles updated in real-time

### Scaling Numbers (Industry Reference)

| System | Items | Users | Embedding Dim | Latency Target |
| --- | --- | --- | --- | --- |
| YouTube | 800M videos | 2B users | 256 | <200ms |
| Spotify | 100M tracks | 600M users | 128 | <150ms |
| Netflix | 17K titles | 230M users | 512 | <100ms |
| Amazon | 350M products | 300M users | 128 | <100ms |
| LinkedIn | 20M jobs | 900M users | 256 | <200ms |

## 10. Summary and Future Directions

### Key Takeaways

1. **Content-based filtering** recommends items based on feature similarity between items a user has liked and candidate items.

2. **Feature representation** is critical — from simple keyword matching to TF-IDF to deep embeddings, the representation determines system quality.

3. **Five main algorithms** exist along a spectrum of complexity:
   - TF-IDF + Cosine Similarity (simplest, most interpretable)
   - Naive Bayes (probabilistic, fast)
   - k-NN (lazy learning, flexible metrics)
   - Decision Trees (rule-based, explainable)
   - Neural Networks (most powerful, least interpretable)

4. **User profile construction** strategy matters: recency-weighted and mean-centered profiles capture evolving tastes better than simple averages.

5. **Evaluation** requires both accuracy metrics (Precision, NDCG) and beyond-accuracy metrics (Diversity, Coverage, Serendipity).

6. **Hybrid systems** combining content-based with collaborative filtering dominate in production.

### Future Directions

- **Large Language Models (LLMs) as recommenders:** Using GPT-4/LLaMA to understand item descriptions and user preferences in natural language
- **Multimodal content understanding:** Jointly modeling text, images, video, and audio for richer item representations
- **Graph Neural Networks:** Encoding item relationships in knowledge graphs for better content understanding
- **Federated content filtering:** Privacy-preserving recommendations where user profiles never leave the device
- **Conversational recommendations:** Interactive systems that refine content preferences through dialogue
- **Causal content filtering:** Moving from correlation ("users who liked X liked Y") to causation ("X caused the user to enjoy Y because of feature Z")

### Recommended Reading

- Ricci, F., Rokach, L., & Shapira, B. (2015). *Recommender Systems Handbook*. Springer.
- Lops, P., de Gemmis, M., & Semeraro, G. (2011). Content-based Recommender Systems: State of the Art and Trends.
- Covington, P., Adams, J., & Sargin, E. (2016). Deep Neural Networks for YouTube Recommendations. *RecSys*.
- Devlin, J. et al. (2019). BERT: Pre-training of Deep Bidirectional Transformers. *NAACL*.